In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

In [3]:
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/small_bench/checkpoints/post_cell_7.pickle

trying: ['REGRESSOR', 'SEP_COMMA', 'ENABLE_BREAKPOINT', 'SHUFFLE_DATA']
me:  2
me:  2
me:  2
me:  2
trying: ['REGRESSOR', 'SEP_COMMA', 'ENABLE_BREAKPOINT', 'SHUFFLE_DATA']
me:  2
me:  2
me:  2
me:  2
trying: ['CONVERT_TO_CAT', 'AUTO_ADJUST_LEARNING_RATE', 'VARIABLE_FILES', 'SEP_DOLLAR']
me:  2
me:  2
me:  2
me:  2
trying: ['CONVERT_TO_CAT', 'AUTO_ADJUST_LEARNING_RATE', 'VARIABLE_FILES', 'SEP_DOLLAR']
me:  2
me:  2
me:  2
me:  2
trying: ['REGRESSOR', 'SEP_COMMA', 'ENABLE_BREAKPOINT', 'SHUFFLE_DATA']
me:  2
me:  2
me:  2
me:  2
trying: ['REGRESSOR', 'SEP_COMMA', 'ENABLE_BREAKPOINT', 'SHUFFLE_DATA']
me:  2
me:  2
me:  2
me:  2
trying: ['CONVERT_TO_CAT', 'AUTO_ADJUST_LEARNING_RATE', 'VARIABLE_FILES', 'SEP_DOLLAR']
me:  2
me:  2
me:  2
me:  2
trying: ['CONVERT_TO_CAT', 'AUTO_ADJUST_LEARNING_RATE', 'VARIABLE_FILES', 'SEP_DOLLAR']
me:  2
me:  2
me:  2
me:  2
trying: ['col']
me:  5
trying: ['Path']
me:  0
trying: ['FASTAI_LEARNING_RATE']
me:  2
trying: ['random']
me:  1
trying: ['shutil']
me: 

In [4]:
%%RecordEvent
%%time
### cell 8 ###

# Optimized for cuDF
import os

# Reset target placeholders
target = ''
target_str = ''
targets = []

# Iterate potential targets in reverse (skip index 0)
for target in reversed(df.columns[1:]):
    # 1) Attempt to convert target column to numeric and drop rows with NaN
    df[target] = pd.to_numeric(df[target], errors='coerce')
    df = df[df[target].notna()]
    target_str = target.replace('/', '-')
    print(f"Target Variable: {target}")

    # 2) Ensure PARAM_DIR and placeholder files exist
    os.makedirs(PARAM_DIR, exist_ok=True)
    for fname in ('cats.txt', 'conts.txt', 'cols_to_delete.txt'):
        open(f"{PARAM_DIR}/{fname}", 'a').close()

    # 3) Drop duplicates and optionally shuffle
    df = df.drop_duplicates()
    if SHUFFLE_DATA:
        df = df.sample(frac=1).reset_index(drop=True)

    # 4) Convert all boolean columns to uint8 in one vectorized step
    bool_cols = [c for c, dt in df.dtypes.items() if dt == 'bool']
    if bool_cols:
        df[bool_cols] = df[bool_cols].astype('uint8')

    # 5) Drop any user‐specified columns in one call
    with open(f"{PARAM_DIR}/cols_to_delete.txt", 'r') as f:
        cols_to_delete = f.read().splitlines()
    df = df.drop(columns=cols_to_delete, errors='ignore')

    # 6) Fill all remaining NaNs with 0
    df = df.fillna(0)

    # 7) Auto‐detect categorical vs continuous variables
    nunique = df.nunique()
    count = df.count()
    cat_mask = (nunique / count) < 0.05
    cats = list(cat_mask[cat_mask].index)
    conts = list(cat_mask[~cat_mask].index)

    # Remove target from variable lists
    if target in cats: cats.remove(target)
    if target in conts: conts.remove(target)

    # 8) Ensure all continuous vars (and target) are floats
    df[conts + [target]] = df[conts + [target]].astype('float64', errors='coerce')

    # 9) If overwriting from files, read back in
    if VARIABLE_FILES:
        with open(f"{PARAM_DIR}/cats.txt", 'r') as f:
            cats = f.read().splitlines()
        with open(f"{PARAM_DIR}/conts.txt", 'r') as f:
            conts = f.read().splitlines()

    # 10) Final pass: coerce continuous cols, fill any NaNs
    df[conts] = df[conts].astype('float64', errors='coerce').fillna(0)

    # 11) Breakpoint logic: detect any failed conversions
    if ENABLE_BREAKPOINT:
        converted = df[conts].astype('float64', errors='coerce')
        # Identify columns with any NaNs after coercion
        failed = {c: converted[c].isna().any() for c in conts}
        for col, did_fail in failed.items():
            if did_fail:
                conts.remove(col)
                if CONVERT_TO_CAT:
                    cats.append(col)
        df[conts] = converted.fillna(0)

# End of cell 8

Target Variable: Profit_processed


ValueError: Expected value of kwarg 'errors' to be one of ['raise', 'ignore']. Supplied value is 'coerce'

In [ ]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/rewritten/o4_mini_high_small/checkpoints/post_cell_8_try_4.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/scratch/jieq/pandax/dias_notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/opt_cell_exec_info_8_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[8], f)


In [ ]:
opt_output = Out.get(4)